<a href="https://colab.research.google.com/github/bala-baskar/LLM_foundations/blob/main/tech_books/Build_a_Large_Language_Model_(From_Scratch)/practice_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Question 1: Implement a Causal Masked Attention Forward Pass
**Problem Statement:**
In a GPT-like Large Language Model, the causal attention mechanism restricts the model from "looking ahead" at future tokens while processing the current token.

Write the `forward` pass for a `CausalAttention` class. You are given:
*   An input tensor `x` of shape `(batch_size, num_tokens, d_in)`.
*   Three initialized linear layers: `self.W_query`, `self.W_key`, and `self.W_value` that map `d_in` to `d_out`.
*   A pre-registered upper-triangular buffer mask `self.mask` (with 1s above the diagonal and 0s elsewhere).
*   A dropout layer `self.dropout`.

**Task:**
Compute the context vectors for the given input sequence.
1. Generate the queries, keys, and values.
2. Compute the unnormalized attention scores via the dot product of queries and keys.
3. **Mask out the future tokens** by replacing the values in the attention scores corresponding to the `self.mask` with `-torch.inf`.
4. Scale the attention scores by the square root of the key dimension and apply softmax to get the attention weights.
5. Apply dropout and return the final context vector matrix.


In [58]:
import torch
from torch import nn

class CausalAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout_rate):
    super().__init__()
    self.W_query = nn.Linear(d_in,d_out,bias=False)
    self.W_key = nn.Linear(d_in,d_out,bias=False)
    self.W_value = nn.Linear(d_in,d_out,bias=False)
    self.mask = torch.triu(input=torch.ones(context_length,context_length),diagonal=1)
    self.dropout = nn.Dropout(dropout_rate)

  def forward(self, x):
    batch, num_tokens, d_in = x.shape
    query = self.W_query(x)
    key = self.W_key(x)
    value = self.W_value(x)
    attention_scores = query @ key.transpose(1,2)
    masked_attn_scores = torch.masked_fill(input=attention_scores,
                                           mask=self.mask == 1,
                                           value=-torch.inf)
    scaled_attn_scores = masked_attn_scores/key.shape[-1]**0.5
    scaled_attn_weights = torch.softmax(scaled_attn_scores,dim=-1)
    scaled_attn_weights_dropout = self.dropout(scaled_attn_weights)
    # Multiply the attention weights by the value matrix
    context_vector = scaled_attn_weights_dropout @ value
    return context_vector

In [59]:
x = torch.randn(4,10,728)
x.shape

torch.Size([4, 10, 728])

In [60]:
ca = CausalAttention(d_in=x.shape[-1],
                     d_out=12,
                     context_length=x.shape[-2],
                     dropout_rate=0.01)

ca(x).shape

torch.Size([4, 10, 12])

### Question 2: Implement a Sliding Window Text Dataset
**Problem Statement:**
When pretraining an LLM, the raw text data must be chunked into overlapping sequences of input and target token IDs.

Write the initialization logic for a `GPTDatasetV1` class that takes a list of `token_ids`, a `max_length` (the context window size), and a `stride`.

**Task:**
Populate two lists, `self.input_ids` and `self.target_ids`, by iterating over the `token_ids` using a sliding window.
*   The `input_chunk` should be a sequence of length `max_length`.
*   The `target_chunk` should be the exact same sequence **shifted one position to the right**.
*   The window should move forward by `stride` steps for each batch.

**Example Input:**
`token_ids =`
`max_length = 4`, `stride = 1`

**Expected Example Output for first 2 iterations:**
`input_ids =`, `target_ids =`
`input_ids =`, `target_ids =`

In [62]:
import urllib.request

# Fetch a sample text (Tiny Shakespeare dataset)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
with urllib.request.urlopen(url) as response:
    full_text = response.read().decode('utf-8')

# Split into words and keep the first 1000
words = full_text.split()
raw_text_1000_words = " ".join(words[:1000])

print(f"Total words extracted: {len(raw_text_1000_words.split())}")
print("\n--- Sample Preview ---")
print(raw_text_1000_words[:500] + "...")

Total words extracted: 1000

--- Sample Preview ---
First Citizen: Before we proceed any further, hear me speak. All: Speak, speak. First Citizen: You are all resolved rather to die than to famish? All: Resolved. resolved. First Citizen: First, you know Caius Marcius is chief enemy to the people. All: We know't, we know't. First Citizen: Let us kill him, and we'll have corn at our own price. Is't a verdict? All: No more talking on't; let it be done: away, away! Second Citizen: One word, good citizens. First Citizen: We are accounted poor citizens...


In [63]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(raw_text_1000_words)
print(len(token_ids))

1395


In [64]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self,token_ids,max_length,stride):
    self.input_ids = []
    self.target_ids = []
    for i in range(0,len(token_ids) - max_length,stride):
      self.input_ids.append(torch.tensor(token_ids[i:i+max_length]))
      self.target_ids.append(torch.tensor(token_ids[i+1:i+max_length+1]))

  def __getitem__(self, index):
    return self.input_ids[index], self.target_ids[index]

  def __len__(self):
    return len(self.input_ids)

In [69]:
dataset = GPTDatasetV1(token_ids=token_ids,
                       max_length=4,
                       stride=4)

data_loader = DataLoader(dataset=dataset,
                         batch_size=4,
                         shuffle=False,
                         drop_last=True,
                         num_workers=0)

next(iter(data_loader))

[tensor([[ 5962, 22307,    25,  7413],
         [  356,  5120,   597,  2252],
         [   11,  3285,   502,  2740],
         [   13,  1439,    25, 40802]]),
 tensor([[22307,    25,  7413,   356],
         [ 5120,   597,  2252,    11],
         [ 3285,   502,  2740,    13],
         [ 1439,    25, 40802,    11]])]

### Question 3: Modify the Output Head for Classification Fine-Tuning
**Problem Statement:**
When adapting a pretrained GPT model for a classification task (like Spam Detection), we must alter its architecture and how we extract predictions. A standard pretrained GPT predicts the next token across the entire vocabulary space, but a classifier only predicts across `num_classes`.

**Task:**
1. Given a base `model` (an instance of `GPTModel`), write a snippet to replace the original output layer `model.out_head` (which currently maps `emb_dim` to `vocab_size`) with a new Linear layer that maps `emb_dim` to `num_classes = 2`.
2. Write a function `get_classification_logits(model, input_batch)` that performs a forward pass. Because of the causal attention mask, the **last token in the sequence accumulates the most information from all previous tokens**. Extract and return **only the logits corresponding to the last token** of the sequence for your classification loss.


In [84]:
from transformers import GPT2Config, GPT2Model

configuration = GPT2Config()

model = GPT2Model(configuration)
model

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [86]:
import torch.nn as nn

# Get the embedding dimension from the model's configuration
emb_dim = model.config.n_embd
num_classes = 2

# Replace the output head
model.out_head = nn.Linear(emb_dim, num_classes)
print(model)

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (out_head): Linear(in_features=768, out_features=2, bias=True)
)


### Question 4: Top-K Sampling for Text Generation
**Problem Statement:**
During text generation, standard greedy decoding (using `argmax`) can lead to repetitive text, while standard temperature scaling might occasionally select grammatically incorrect, highly improbable tokens. Top-k sampling restricts the next-token selection to only the `k` most probable tokens.

**Task:**
Given a 1D tensor of `logits` (representing the next-token prediction output of a model) and an integer `top_k`:
1. Use PyTorch to find the values of the top `top_k` logits.
2. Create a `new_logits` tensor where any value in the original `logits` that is **less than the smallest value in your top-k set** is replaced with `-torch.inf`.
3. Apply softmax on `new_logits` to return the final probability distribution.

In [104]:
# Create random logits tensor
torch.manual_seed(123)
logits = torch.randn(10)
print("Logits:\n",logits)
# Get top K based on large values
topk_indices = torch.topk(input=logits,k=3).indices
print("Top K values:\n",logits[topk_indices])
# Create new logits to mask smaller values with -inf
new_logits = torch.ones(logits.shape) * (-torch.inf)
for i in range(len(logits)):
  if i in topk_indices:
    new_logits[i] = logits[i]
print("New Logits:\n",new_logits)
# Apply softmax to get probability
probs = torch.softmax(new_logits,dim=-1)
print("Probability distributed for Top k tokens:\n",probs)

Logits:
 tensor([-0.1115,  0.1204, -0.3696, -0.2404, -1.1969,  0.2093, -0.9724, -0.7550,
         0.3239, -0.1085])
Top K values:
 tensor([0.3239, 0.2093, 0.1204])
New Logits:
 tensor([  -inf, 0.1204,   -inf,   -inf,   -inf, 0.2093,   -inf,   -inf, 0.3239,
          -inf])
Probability distributed for Top k tokens:
 tensor([0.0000, 0.3013, 0.0000, 0.0000, 0.0000, 0.3293, 0.0000, 0.0000, 0.3693,
        0.0000])


In [105]:
top_k = 3
# Create random logits tensor
torch.manual_seed(123)
logits = torch.randn(10)
# 1. Find the top k values
top_values, top_indices = torch.topk(logits, top_k)

# 2. The smallest value in our top-k set is the last element (since topk sorts by default)
min_top_value = top_values[-1]

# 3. Create new_logits by masking out anything smaller than min_top_value
new_logits_opt = torch.where(logits < min_top_value, torch.tensor(-torch.inf), logits)
print("Optimized New Logits:\n", new_logits_opt)

# 4. Apply softmax to get final probability distribution
probs_opt = torch.softmax(new_logits_opt, dim=-1)
print("Optimized Probability distribution:\n", probs_opt)


Optimized New Logits:
 tensor([  -inf, 0.1204,   -inf,   -inf,   -inf, 0.2093,   -inf,   -inf, 0.3239,
          -inf])
Optimized Probability distribution:
 tensor([0.0000, 0.3013, 0.0000, 0.0000, 0.0000, 0.3293, 0.0000, 0.0000, 0.3693,
        0.0000])
